In [28]:
from datetime import datetime
from difflib import unified_diff
from pathlib import Path
from typing import Annotated, Literal, Self

import instructor
import pandas as pd
from google.generativeai import GenerativeModel
from pydantic import (
    AfterValidator,
    BaseModel,
    Field,
    model_validator,
    validate_call,
)

from dreamai.ai import ModelName, system_message, user_message

ask_gemini = instructor.from_gemini(
    GenerativeModel(model_name=ModelName.GEMINI_PRO_EXP)
)


%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
DEFAULT_VERSION = 1.0
CONTEXT_LINES = 0

ASSERTION_CATEGORIES = Literal[
    "Presentation Format",
    "Example Demonstration",
    "Workflow Description",
    "Count",
    "Inclusion",
    "Exclusion",
    "Qualitative Assessment",
    "Other",
]


class AssertionConcept(BaseModel):
    concept: str
    category: ASSERTION_CATEGORIES
    source: str


class ChangeRecord(BaseModel):
    from_version: float
    to_version: float
    diff: str
    description: str = ""
    timestamp: datetime = Field(default_factory=datetime.now)


def load_template(template: str | Path) -> str:
    if isinstance(template, Path):
        return template.read_text()
    if template.strip().endswith(".txt"):
        return Path(template).read_text()
    return template


TemplateType = Annotated[str, AfterValidator(load_template)]


class PromptTemplate(BaseModel):
    name: str
    template: TemplateType
    version: float = DEFAULT_VERSION
    description: str = ""
    original_template: TemplateType = ""
    created_at: datetime = Field(default_factory=datetime.now)
    change_history: list[ChangeRecord] = []

    @model_validator(mode="after")
    def validate_original_template(self) -> Self:
        self.original_template = self.original_template or self.template
        return self

    @property
    def current_template(self):
        return self.template

    def render(self, **kwargs) -> str:
        try:
            return self.template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required placeholder: {e}")

    @validate_call
    def update(
        self,
        new_template: TemplateType,
        new_version: float | None = None,
        description: str = "",
    ):
        if new_template == self.template:
            return
        if new_version is None:
            num_decimal_paces = str(self.version)[::-1].find(".")
            new_version = round(
                self.version + 10**-num_decimal_paces, num_decimal_paces
            )
        diff = list(
            unified_diff(
                self.template.splitlines(keepends=True),
                new_template.splitlines(keepends=True),
                fromfile=f"v{self.version}",
                tofile=f"v{new_version}",
                n=CONTEXT_LINES,
            )
        )
        self.change_history.append(
            ChangeRecord(
                from_version=self.version,
                to_version=new_version,  # type: ignore
                description=description,
                diff="".join(diff),
            )
        )
        self.template = new_template
        self.version = new_version  # type: ignore
        self.description = description
        with open(f"{self.name}.json", "w") as f:
            f.write(self.model_dump_json(indent=2))

    def reset_to_original(self):
        desc = "Reset to original template"
        self.template = self.original_template
        self.version = DEFAULT_VERSION
        self.description = desc
        self.change_history.append(
            ChangeRecord(
                from_version=self.version,
                to_version=DEFAULT_VERSION,
                description=desc,
                diff="",
            )
        )

In [ ]:
template = PromptTemplate(
    name="assertion_concept_prompt",
    template="You are a linguistic analysis tool designed to extract key components from sentences.",
)

template.update(
    new_template="""\
You are a linguistic analysis tool designed to extract key components from sentences. Your task is to identify and extract the following elements from a given sentence:

1. Noun: The main content noun(s) or noun phrase(s) in the sentence.
2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.
3. Object: The main recipient(s) of the action, if present.
4. Verb: The primary action(s) or state(s) of being in the sentence.
5. Adjective: Key describing word(s), if present.

Please ensure to extract all the elements accurately and concisely.
""",
)

template.update(new_template="v2.txt")

In [49]:
print(template.template)

You are a linguistic analysis tool designed to extract key components from sentences. Your task is to identify and extract the following elements from a given sentence:

1. Noun: The main content noun(s) or noun phrase(s) in the sentence.
2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.
3. Object: The main recipient(s) of the action, if present.
4. Verb: The primary action(s) or state(s) of being in the sentence.
5. Adjective: Key describing word(s), if present.

For each sentence provided, you should:
- Analyze the sentence structure.
- Identify each of the above components if they exist in the sentence.
- Focus on content words; avoid pronouns, articles, and other function words.
- If a component is not present, leave its list empty.
- Include only the most important items for each component, up to a maximum of 5 per component.


In [50]:
template.change_history[0].model_dump()

{'from_version': 1.0,
 'to_version': 1.1,
 'diff': '--- v1.0\n+++ v1.1\n@@ -1 +1,9 @@\n-You are a linguistic analysis tool designed to extract key components from sentences.+You are a linguistic analysis tool designed to extract key components from sentences. Your task is to identify and extract the following elements from a given sentence:\n+\n+1. Noun: The main content noun(s) or noun phrase(s) in the sentence.\n+2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.\n+3. Object: The main recipient(s) of the action, if present.\n+4. Verb: The primary action(s) or state(s) of being in the sentence.\n+5. Adjective: Key describing word(s), if present.\n+\n+Please ensure to extract all the elements accurately and concisely.\n',
 'description': '',
 'timestamp': datetime.datetime(2024, 8, 5, 19, 37, 7, 830729)}

In [51]:
print(template.change_history[0].diff)

--- v1.0
+++ v1.1
@@ -1 +1,9 @@
-You are a linguistic analysis tool designed to extract key components from sentences.+You are a linguistic analysis tool designed to extract key components from sentences. Your task is to identify and extract the following elements from a given sentence:
+
+1. Noun: The main content noun(s) or noun phrase(s) in the sentence.
+2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.
+3. Object: The main recipient(s) of the action, if present.
+4. Verb: The primary action(s) or state(s) of being in the sentence.
+5. Adjective: Key describing word(s), if present.
+
+Please ensure to extract all the elements accurately and concisely.



In [52]:
print(template.model_dump_json(indent=2))

{
  "name": "assertion_concept_prompt",
  "template": "You are a linguistic analysis tool designed to extract key components from sentences. Your task is to identify and extract the following elements from a given sentence:\n\n1. Noun: The main content noun(s) or noun phrase(s) in the sentence.\n2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.\n3. Object: The main recipient(s) of the action, if present.\n4. Verb: The primary action(s) or state(s) of being in the sentence.\n5. Adjective: Key describing word(s), if present.\n\nFor each sentence provided, you should:\n- Analyze the sentence structure.\n- Identify each of the above components if they exist in the sentence.\n- Focus on content words; avoid pronouns, articles, and other function words.\n- If a component is not present, leave its list empty.\n- Include only the most important items for each component, up to a maximum of 5 per component.",
  "version": 1.2,
  "description": "",
  "o

In [53]:
concepts = ask_gemini.create(
    messages=[
        system_message(Path("assertion_concept_prompt.txt").read_text()),
        user_message(template.model_dump_json(indent=2)),
    ],  # type: ignore
    response_model=list[AssertionConcept],
)

In [54]:
concepts

[AssertionConcept(concept="The response should be a JSON object with keys for 'Noun', 'Subject', 'Object', 'Verb', and 'Adjective'.", category='Presentation Format', source='Your task is to identify and extract the following elements from a given sentence:\n\n1. Noun: The main content noun(s) or noun phrase(s) in the sentence.\n2. Subject: The key person(s), thing(s), or idea(s) performing the action or being described.\n3. Object: The main recipient(s) of the action, if present.\n4. Verb: The primary action(s) or state(s) of being in the sentence.\n5. Adjective: Key describing word(s), if present.'),
 AssertionConcept(concept='Each key in the JSON object should correspond to a list of identified words or phrases.', category='Presentation Format', source='For each sentence provided, you should:\n- Analyze the sentence structure.\n- Identify each of the above components if they exist in the sentence.'),
 AssertionConcept(concept='The response should analyze the sentence structure to ide

In [55]:
concepts_df = pd.DataFrame(
    [
        {"concept": c.concept, "category": c.category, "source": c.source}
        for c in concepts  # type: ignore
    ]
)
concepts_df

,concept,category,source
0,The response should be a JSON object with keys...,Presentation Format,Your task is to identify and extract the follo...
1,Each key in the JSON object should correspond ...,Presentation Format,"For each sentence provided, you should:\n- Ana..."
2,The response should analyze the sentence struc...,Workflow Description,"For each sentence provided, you should:\n- Ana..."
3,The response should focus on content words and...,Exclusion,"Focus on content words; avoid pronouns, articl..."
4,"If a component is not present in the sentence,...",Presentation Format,"If a component is not present, leave its list ..."
5,The response should include only the most impo...,Qualitative Assessment,Include only the most important items for each...
6,"Each component (Noun, Subject, Object, Verb, A...",Count,Include only the most important items for each...


In [56]:
concepts[3].model_dump()

{'concept': 'The response should focus on content words and avoid pronouns, articles, and other function words.',
 'category': 'Exclusion',
 'source': 'Focus on content words; avoid pronouns, articles, and other function words.'}